In [ ]:
import numpy as np
import pandas as pd

In [ ]:
quan = pd.read_excel(f"/kaggle/input/widsdatathon2025/TRAIN_NEW/TRAIN_QUANTITATIVE_METADATA_new.xlsx")
quan_features_name = quan.columns
quan.head()

In [ ]:
cate = pd.read_excel(f"/kaggle/input/widsdatathon2025/TRAIN_NEW/TRAIN_CATEGORICAL_METADATA_new.xlsx")
cate_features_name = cate.columns
cate.head()

In [ ]:
func = pd.read_csv(f"/kaggle/input/widsdatathon2025/TRAIN_NEW/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv")
func_features_name = func.columns
func.head()

In [ ]:
solution = pd.read_excel(f"/kaggle/input/widsdatathon2025/TRAIN_NEW/TRAINING_SOLUTIONS.xlsx")
solution_features_name = solution.columns
solution.head()

In [ ]:
'''
def filter_corr(df):
    id = ["participant_id"]
    # Compute correlation matrix
    correlation_matrix = df.drop(columns=id).corr().abs()
    
    # Set correlation threshold
    threshold = 0.7
    
    # Identify columns to drop
    upper_triangle = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > threshold)]

    return to_drop
''' 

In [ ]:
'''
quan_corr = filter_corr(quan)
quan_corr
'''

In [ ]:
'''cate_corr = filter_corr(cate)
cate_corr'''

In [ ]:
'''func_corr = filter_corr(func)
func_corr'''

In [ ]:
'''print(quan_features_name)
print(cate_features_name)
print(func_features_name)'''

In [ ]:

def get_feats(mode='TRAIN'):
    """
    Load data for the specified mode (TRAIN or TEST).
    """
    # Load quantitative metadata
    feats = pd.read_excel(f"/kaggle/input/widsdatathon2025/TRAIN_NEW/TRAIN_QUANTITATIVE_METADATA_new.xlsx")
    
    # Load categorical metadata
    if mode == 'TRAIN':
        cate = pd.read_excel(f"/kaggle/input/widsdatathon2025/TRAIN_NEW/TRAIN_CATEGORICAL_METADATA_new.xlsx")
    else:
        cate = pd.read_excel(f"/kaggle/input/widsdatathon2025/{mode}/{mode}_CATEGORICAL.xlsx")
    
    # Merge quantitative and categorical data
    feats = pd.merge(feats, cate, on='participant_id', how='left')
    
    # Load functional connectome matrices
    func = pd.read_csv(f"/kaggle/input/widsdatathon2025/TRAIN_NEW/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv")
    feats = pd.merge(feats, func, on='participant_id', how='left')
    
    # Load training solutions (only for TRAIN mode)
    if mode == 'TRAIN':
        solution = pd.read_excel("/kaggle/input/widsdatathon2025/TRAIN_NEW/TRAINING_SOLUTIONS.xlsx")
        feats = pd.merge(feats, solution, on='participant_id', how='left')
    
    return feats

# Load training and test data
print("Loading data...")
train = get_feats(mode='TRAIN')
test = get_feats(mode='TEST')

# Display the first few rows of the training data
train.head()

In [ ]:
train.set_index('participant_id',inplace=True)
test.set_index('participant_id',inplace=True)

In [ ]:
test.shape

In [ ]:
targets = ['ADHD_Outcome','Sex_F']
features = test.columns

print("features: ")
print(features)
print("targets: ")
print(targets)

In [ ]:
print(train.isnull().sum()[train.isnull().sum() > 0]) 
print(test.isnull().sum()[test.isnull().sum() > 0])    

In [ ]:
train.dropna(inplace=True)
#test.dropna(inplace=True)

In [ ]:
print(train.isnull().sum()[train.isnull().sum() > 0]) 
#print(test.isnull().sum()[test.isnull().sum() > 0])

In [ ]:
drop_columns = [
    'APQ_P_APQ_P_PP',
    'SDQ_SDQ_Difficulties_Total',
    'SDQ_SDQ_Externalizing',
    'SDQ_SDQ_Generating_Impact',
    'SDQ_SDQ_Hyperactivity',
    'SDQ_SDQ_Internalizing',
    'SDQ_SDQ_Peer_Problems',
    'MRI_Track_Scan_Location',
    'Barratt_Barratt_P2_Occ',
    '128throw_129thcolumn'
]

train.drop(columns = drop_columns, errors='ignore', inplace=True)
test.drop(columns = drop_columns, errors='ignore', inplace=True)

In [ ]:
train.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

exclude_cols = ['ADHD_Outcome', 'Sex_F']
# standarization.
num_cols = [col for col in train.select_dtypes(include=['number']).columns if col not in exclude_cols]
scaler = StandardScaler()
train[num_cols] = scaler.fit_transform(train[num_cols])
test[num_cols] = scaler.transform(test[num_cols])


In [ ]:
train[num_cols]

In [ ]:
test[num_cols]

In [ ]:
train.columns

In [ ]:
# Separate features and target variables
X = train.drop(['participant_id', 'ADHD_Outcome', 'Sex_F'], axis=1, errors='ignore')

# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(exclude=['object']).columns.tolist()

print("categorical features: ")
print(len(categorical_features))
print("numerical features: ")
print(len(numerical_features))

In [ ]:
features = X.columns
log_features = [f for f in features if (train[f] >= 0).all() and scipy.stats.skew(train[f]) > 0]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import f1_score

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train.drop(targets,axis=1), 
                                                    train[targets], 
                                                    test_size=0.50, 
                                                    random_state=40)
model = MultiOutputClassifier(make_pipeline(
                        
                              ColumnTransformer([('imputer',SimpleImputer(),features)],
                                               remainder='passthrough',
                                               verbose_feature_names_out=False).set_output(transform='pandas'),
                              ColumnTransformer([('log', 
                                                 FunctionTransformer(np.log1p), log_features)],
                                                 remainder='passthrough'),
                              
                            MinMaxScaler(),    
                              
                            RidgeClassifier(alpha=100)))
print("training")
model.fit(X_train,y_train)
print("predicting")
y_pred = model.predict(X_test)
print('f1: ', f1_score(y_test,y_pred,average='micro'))

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'alpha': [0.1, 1, 10, 50, 100, 200, 500]}
ridge = RidgeClassifier()

grid_search = GridSearchCV(ridge, param_grid, scoring='f1_micro', cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

best_alpha = grid_search.best_params_['alpha']
print("Best alpha:", best_alpha)

model = MultiOutputClassifier(make_pipeline(
    ColumnTransformer([('imputer', SimpleImputer(), features)], remainder='passthrough').set_output(transform='pandas'),
    ColumnTransformer([('log', FunctionTransformer(np.log1p), log_features)], remainder='passthrough'),
    MinMaxScaler(),
    RidgeClassifier(alpha=best_alpha)  
))

print("training")
model.fit(X_train,y_train)
print("predicting")
y_pred = model.predict(X_test)
print('f1: ', f1_score(y_test,y_pred,average='micro'))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=500, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

from sklearn.metrics import f1_score
print("F1 Score:", f1_score(y_test, y_pred_rf, average='micro'))

In [ ]:
from sklearn.ensemble import StackingClassifier

base_models = [
    ('ridge', RidgeClassifier(alpha=best_alpha)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
]

meta_model = LogisticRegression()

stacking_model = MultiOutputClassifier(
    StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)
)

stacking_model.fit(X_train, y_train)

y_pred_stack = stacking_model.predict(X_test)

print("F1 Score:", f1_score(y_test, y_pred_stack, average='micro'))

In [ ]:
y_pred = model.predict(test)

In [ ]:
sub = pd.read_excel('/kaggle/input/widsdatathon2025/SAMPLE_SUBMISSION.xlsx')
sub['ADHD_Outcome'] = y_pred[:304, 0]
sub['Sex_F'] = y_pred[:304, 1]
sub.to_csv('submission.csv',index=False)